# Volume 13, Chapter 8: Small gaps between primes (Python companion)

Exercises first, solutions at the end. We test admissibility, count prime constellations against the Hardy–Littlewood prediction, optimize the one-variable Goldston–Pintz–Yıldırım weights as a generalized eigenvalue problem, and watch the GPY sieve weights at work on actual integers. The computer science application: **backtracking search with constraint pruning**, used to find the narrowest admissible $k$-tuples. Requires `numpy`.

In [ ]:
import math, time, itertools
import numpy as np

def prime_sieve(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return s

def small_primes(n):
    return [int(p) for p in np.nonzero(prime_sieve(n))[0]]

def nu(H, p):
    return len({h % p for h in H})

def is_admissible(H):
    return all(nu(H, p) < p for p in small_primes(len(H)))

## Exercises

**P8.1 (★) Admissible or not.** Decide which of $\{0,2,4\}$, $\{0,2,6,8,12\}$, $\{0,4,6,10,12,16\}$, $\{0,2,6,8,12,18,20\}$ are admissible, and for an inadmissible one name the prime that blocks it. Check that the $k$ smallest primes larger than $k$ form an admissible tuple for $k\le200$ (Exercise 8.1), and compare its diameter with $k\log k$.

**P8.2 (★) Constellations.** Up to $x=10^7$, count the $n$ with $n+\mathcal H$ all prime for $\mathcal H=\{0,2\},\{0,2,6\},\{0,4,6\},\{0,2,6,8\},\{0,4,6,10\},\{0,2,6,8,12\}$, and compare with $\mathfrak S(\mathcal H)\int_2^x\frac{\mathrm dt}{(\log t)^k}$. Compute $\mathfrak S(\mathcal H)$ as a product over $p<10^6$ and estimate the truncation error.

**P8.3 (★★) The best one-variable weight.** For $F(\mathbf t)=P(1-\sum t_i)$ with $P$ a polynomial of degree at most $d$, Proposition 8.9 turns $\frac{\sum_mJ_k^{(m)}(F)}{I_k(F)}$ into a ratio of two quadratic forms in the coefficients of $P$. Write the two Gram matrices, find the maximum by a generalized eigenvalue problem, and compare with the value $\frac{2k(2\ell+1)}{(\ell+1)(k+2\ell+1)}$ of $P=x^\ell$ for the best $\ell$. For which $k$ does the best ratio first exceed $2$, and how large must $\theta$ be for Corollary 8.8 to give two primes in $n+\mathcal H$ with $k=6$, $7$?

**P8.4 (★★) The weights at work.** Take $\mathcal H=\{0,2,6\}$, $N=10^6$, $R=N^{\vartheta}$ with $\vartheta\in\{\frac14,\frac13,\frac25\}$, and the GPY weights $w(n)=\bigl(\sum_{d\mid P(n),\,d<R}\mu(d)(\log\frac Rd)^{k+\ell}\bigr)^2$, $P(n)=\prod(n+h_i)$, $\ell=1$. Compute the weighted average number of primes among $n,n+2,n+6$ over $N\le n<2N$, compare it with the unweighted average and with the asymptotic prediction $\frac{\log R}{\log N}\cdot\frac{2k(2\ell+1)}{(\ell+1)(k+2\ell+1)}$, and describe what the weights favour.

**P8.5 (★★) Narrowest admissible tuples (computer science).** Let $H(k)$ be the smallest diameter $h_k-h_1$ of an admissible $k$-tuple. Find $H(k)$ for $2\le k\le 12$ by a backtracking search over even offsets, pruning a partial tuple as soon as it covers every class modulo some prime $p\le k$. Count the nodes visited with and without the pruning, and compare $H(k)$ with the diameter of the tuple of P8.1.

## Solutions

### P8.1

In [ ]:
for H in [[0,2,4],[0,2,6,8,12],[0,4,6,10,12,16],[0,2,6,8,12,18,20]]:
    bad = [p for p in small_primes(len(H)) if nu(H, p) == p]
    print(H, "admissible" if not bad else f"blocked by p = {bad}")

def first_primes_tuple(k):
    out, n = [], k + 1
    while len(out) < k:
        if all(n % p for p in range(2, int(n ** 0.5) + 1)): out.append(n)
        n += 1
    return out

ok = all(is_admissible(first_primes_tuple(k)) for k in range(2, 201))
print("first-primes tuples admissible for k <= 200:", ok)
for k in [5, 10, 50, 105, 200]:
    T = first_primes_tuple(k)
    print(f"k={k:3d}  diameter {T[-1]-T[0]:5d}   k log k = {k*math.log(k):7.0f}")

$\{0,2,4\}$ covers all three classes modulo $3$, so one of $n,n+2,n+4$ is divisible by $3$: the only prime triple of this shape is $3,5,7$. The other three are admissible; the last is the $7$-tuple of Theorem 8.10(b). The tuple of the first $k$ primes beyond $k$ is admissible because none of its elements is divisible by a prime $p\le k$, so the class $0$ is missed for every such $p$. Its diameter is of the order of $k\log k$ by the prime number theorem, about $1.3\,k\log k$ in this range. For $k=105$ it is $636$, while the narrowest admissible $105$-tuple, found by Engelsma, has diameter $600$: P8.5 shows how a search finds such savings.

### P8.2

In [ ]:
X = 10 ** 7
isp = prime_sieve(X + 20)
plist = np.nonzero(prime_sieve(10 ** 6))[0]

def sing(H):
    v = 1.0; k = len(H)
    for p in plist:
        v *= (1 - nu(H, int(p)) / p) / (1 - 1 / p) ** k
    return v

def li_k(x, k, steps=200000):
    # integral of dt/(log t)^k from 2 to x, with t = e^u (trapezoid in u)
    u = np.linspace(math.log(2), math.log(x), steps + 1)
    f = np.exp(u) / u ** k
    return float((f[0] / 2 + f[1:-1].sum() + f[-1] / 2) * (u[1] - u[0]))

for H in [[0,2],[0,2,6],[0,4,6],[0,2,6,8],[0,4,6,10],[0,2,6,8,12]]:
    m = np.ones(X + 1, dtype=bool)
    for h in H: m &= isp[h:h + X + 1]
    c = int(m.sum()); S = sing(H); pred = S * li_k(X, len(H))
    print(f"{str(H):18s} count {c:7d}   S = {S:.4f}   prediction {pred:9.1f}   ratio {c/pred:.4f}")

Up to $10^7$ the ratios are $1.004$ for twins, $0.994$ and $1.010$ for the two triple shapes, $1.04$ and $0.96$ for the two quadruple shapes, and $0.92$ for the $5$-tuple, which has only $160$ members: the fewer the members, the larger the fluctuations, but the prediction is right within the noise. The truncation error of $\mathfrak S$ is small: for $p>k$ the factor is $1+O(k^2/p^2)$, and $\sum_{p>10^6}k^2/p^2\approx\frac{k^2}{10^6\log10^6}$, below $10^{-5}$ for $k\le5$. The two triple shapes have the same singular series (they are mirror images, $\{0,4,6\}=6-\{0,2,6\}$), and their counts differ by $1.6\%$ here and by less than $0.1\%$ at $10^8$ (Figure 8.1).

### P8.3

In [ ]:
def beta(a, b): return math.gamma(a) * math.gamma(b) / math.gamma(a + b)

def gram(k, d):
    # P(x) = sum c_i x^i ; I = c^T A c, sum_m J^(m) = c^T B c  (Proposition 8.9)
    A = np.array([[beta(i + j + 1, k) / math.factorial(k - 1) for j in range(d + 1)] for i in range(d + 1)])
    B = np.array([[k * beta(i + j + 3, k - 1) / math.factorial(k - 2) / ((i + 1) * (j + 1))
                   for j in range(d + 1)] for i in range(d + 1)])
    return A, B

def best_ratio(k, d=8):
    A, B = gram(k, d)
    L = np.linalg.cholesky(A); Li = np.linalg.inv(L)
    w, v = np.linalg.eigh(Li @ B @ Li.T)
    c = Li.T @ v[:, -1]
    return w[-1], c / c[np.argmax(abs(c))]

def monomial_ratio(k, l): return 2 * k * (2 * l + 1) / ((l + 1) * (k + 2 * l + 1))

for k in [2, 3, 4, 5, 6, 7, 10, 20, 50, 100]:
    r, c = best_ratio(k)
    lbest = max(range(0, 60), key=lambda l: monomial_ratio(k, l))
    print(f"k={k:3d}  best degree-8 ratio {r:.4f}   best x^l: l={lbest:2d}, {monomial_ratio(k, lbest):.4f}")

In [ ]:
for k in [6, 7]:
    r, _ = best_ratio(k)
    print(f"k={k}: two primes need theta > 2/ratio = {2/r:.4f}")
# sanity check of the matrices against Proposition 8.9 with P = x^1, k = 7
A, B = gram(7, 3); c = np.array([0, 1, 0, 0.0])
print("x^1, k=7:", (c @ B @ c) / (c @ A @ c), monomial_ratio(7, 1))

The Gram matrices come straight from Proposition 8.9: with $P=\sum c_ix^i$, $I_k=\sum c_ic_j\frac{B(i+j+1,k)}{(k-1)!}$ by Euler's Beta integral, and $Q=\sum\frac{c_i}{i+1}x^{i+1}$ gives $J_k^{(m)}=\sum\frac{c_ic_j}{(i+1)(j+1)}\frac{B(i+j+3,k-1)}{(k-2)!}$. Maximizing a ratio of quadratic forms $\frac{c^TBc}{c^TAc}$ is the largest eigenvalue of $L^{-1}BL^{-T}$ where $A=LL^T$ (Cholesky), the same Rayleigh-quotient trick that Chapter 9 uses for many-variable $F$.

The best ratio crosses $2$ already at $k=6$ ($2.084$, against $2$ for the best monomial, which never exceeds $2$ for $k\le6$), so $k=6$ needs $\theta>2/2.084\approx0.960$ and $k=7$ needs $\theta>2/2.184\approx0.916$, while $x^1$ at $k=7$ gives exactly $\frac{2\cdot7\cdot3}{2\cdot10}=2.1$, i.e. $\theta>\frac{20}{21}$ as in Theorem 8.10(b). The check of $P=x$ at $k=7$ confirms the matrices. For large $k$ the optimum creeps towards $4$ but stays below: at $k=100$ it is $3.473$ against $3.303$ for the best monomial, and Exercise 8.3 shows that it is at most $\frac{4k}{k-1}$, so one variable can never go much beyond $4$. One variable is not enough at level $\frac12$, which is the whole point of Chapter 9.

### P8.4

In [ ]:
H = [0, 2, 6]; k = len(H); l = 1
N = 10 ** 6
n = np.arange(N, 2 * N, dtype=np.int64)
isp2 = prime_sieve(2 * N + 10)
nprimes = sum(isp2[N + h: 2 * N + h].astype(np.int64) for h in H)

def mobius_upto(M):
    mu = np.ones(M + 1, dtype=np.int64); mu[0] = 0
    for p in small_primes(M):
        mu[p::p] *= -1; mu[p * p::p * p] = 0
    return mu

for vt in [0.25, 1 / 3, 0.4]:
    R = N ** vt
    mu = mobius_upto(int(R) + 1)
    lam = np.zeros(N)
    for d in range(1, int(R) + 1):
        if d >= R or mu[d] == 0: continue
        r = (n % d) * ((n + 2) % d) % d * ((n + 6) % d) % d
        lam += np.where(r == 0, mu[d] * math.log(R / d) ** (k + l), 0.0)
    w = lam ** 2
    weighted = float((nprimes * w).sum() / w.sum())
    pred = math.log(R) / math.log(N) * monomial_ratio(k, l)
    print(f"R = N^{vt:.3f} = {R:7.1f}: weighted {weighted:.4f}   unweighted {nprimes.mean():.4f}   prediction {pred:.4f}")
    if vt == 0.4:
        for j in range(4):
            sel = nprimes == j
            print(f"   {j} primes among n, n+2, n+6: share of n {sel.mean():.4f},  share of weight {w[sel].sum()/w.sum():.4f}")

Unweighted, an $n$ in $[N,2N[$ has on average $0.211$ primes among $n,n+2,n+6$. The weights raise this to $0.367$, $0.470$ and $0.560$ for the three values of $R$, within $7\%$ of the asymptotic predictions $0.375$, $0.5$ and $0.6$, although $\log R$ is only between $3.5$ and $5.5$ and the error terms in Theorem 8.7 are of relative size $1/\log R$. With $R=N^{2/5}$, the $n$ with two or three primes carry almost $10\%$ of the weight, against $2.4\%$ of the $n$. The breakdown by the number of primes shows what the weights do: they move mass away from the $n$ for which $P(n)$ has small prime factors (these get $\lambda$ close to $0$ by cancellation) towards the $n$ with all three entries free of small factors, where two or three primes are far more common. The weights are a smooth version of "sieve out small primes from $n(n+2)(n+6)$", which is exactly the GPY idea; making the weighted average exceed $1$ is what the whole chapter is about, and at level $\frac12$ one variable gets only up to about $\frac14\cdot4=1$.

### P8.5

In [ ]:
def narrowest(k, prune_primes=None):
    ps = small_primes(k) if prune_primes is None else prune_primes
    full = small_primes(k)
    nodes = [0]
    def ok_partial(T):
        return all(len({t % p for t in T}) < p for p in ps)
    def search(D):
        # tuples 0 = h_1 < ... < h_k = D, even offsets
        mids = list(range(2, D, 2))
        def rec(T, start, need):
            nodes[0] += 1
            if not ok_partial(T): return None
            if need == 0:
                return T if all(len({t % p for t in T}) < p for p in full) else None
            for i in range(start, len(mids) - need + 1):
                res = rec(T + [mids[i]], i + 1, need - 1)
                if res: return res
            return None
        return rec([0, D], 0, k - 2)
    D = 2
    while True:
        T = search(D)
        if T: return D, sorted(T), nodes[0]
        D += 2

for k in range(2, 13):
    t0 = time.time(); D, T, nd = narrowest(k)
    print(f"k={k:2d}  H(k)={D:3d}  {T}  nodes {nd:8d}  {time.time()-t0:.2f}s   first-primes diameter {first_primes_tuple(k)[-1]-first_primes_tuple(k)[0]}")

In [ ]:
for k in [8, 9, 10]:
    t0 = time.time(); D1, _, nd1 = narrowest(k, prune_primes=[])
    t1 = time.time(); D2, _, nd2 = narrowest(k)
    t2 = time.time()
    print(f"k={k}: no pruning {nd1:9d} nodes ({t1-t0:.2f}s)   pruning {nd2:7d} nodes ({t2-t1:.2f}s)")

The search returns $H(k)=2,6,8,12,16,20,26,30,32,36,42$ for $k=2,\dots,12$, the known values (OEIS A008407). Up to $k=10$ the first-primes tuple of P8.1, shifted to start at $0$, is already optimal; from $k=11$ on the search wins ($36$ against $40$, $42$ against $46$). Only even offsets are needed, since an admissible tuple with $k\ge2$ cannot contain both parities (it would cover both classes modulo $2$). Pruning tests each partial tuple modulo $3,5,\dots$ as soon as an element is added, so a branch that already covers all three classes modulo $3$ dies at once instead of being extended in all possible ways; the node counts show the saving grows quickly with $k$. This is the standard shape of a constraint search: choose variables in order, check the constraints that only involve the chosen ones, and backtrack. For large $k$ (Polymath8 went up to $k$ in the tens of thousands) one switches from exhaustive search to clever constructions, a sieve over an interval that removes one residue class for each small prime, followed by local improvements. The narrowest $105$-tuple has diameter $600$, which is how Maynard's $k=105$ becomes "gaps at most $600$" in Chapter 9.